# Graph Coloring For Pauli-String Ordering

This notebook is a small teaching companion for `group_coloring_ordering_demo.py`.

The idea is:

- one Pauli term becomes one graph node
- one graph edge means two Pauli strings do **not** commute
- one graph color is therefore a pairwise-commuting group

For H2/STO-3G, this shows why the apparent `14!` ordering problem is much smaller once commuting swaps are ignored.

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = next(path for path in [cwd, *cwd.parents] if (path / "analysis" / "ordering.py").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis.examples.group_coloring_ordering_demo import (
    build_greedy_coloring_groups,
    flatten_groups,
    format_dense_pauli,
    group_is_pairwise_commuting,
    h2_sto3g_jw_terms,
    noncommuting_pairs,
)
from analysis.ordering import reorder_paulis

## 1. Load The Toy H2 Pauli Terms

The identity term is omitted because it commutes with everything and only contributes a global phase.

In [ ]:
terms = h2_sto3g_jw_terms(include_identity=False)

for index, (pauli_string, coefficient) in enumerate(terms.items()):
    print(f"{index:2d}: {pauli_string:4s}  {format_dense_pauli(pauli_string):15s}  {coefficient:+.10e}")

## 2. Build The Noncommutation Graph

`ordering.py` calls this helper `create_commutativity_graph`, but the current implementation adds an edge when two Pauli strings **do not** commute. That makes it the graph we want for coloring.

In [ ]:
bad_pairs = noncommuting_pairs(terms)
total_pairs = len(terms) * (len(terms) - 1) // 2

print(f"terms                : {len(terms)}")
print(f"all unordered pairs  : {total_pairs}")
print(f"commuting pairs      : {total_pairs - len(bad_pairs)}")
print(f"noncommuting pairs   : {len(bad_pairs)}")

for left, right in bad_pairs:
    print(f"{format_dense_pauli(left[0]):15s} <-> {format_dense_pauli(right[0])}")

## 3. Color The Graph

Adjacent nodes cannot share a color. Since edges mean noncommutation, every color group is pairwise commuting.

In [ ]:
graph, coloring, groups = build_greedy_coloring_groups(terms)

print(f"graph nodes : {graph.number_of_nodes()}")
print(f"graph edges : {graph.number_of_edges()}")
print(f"colors      : {len(groups)}")

for color, group in enumerate(groups):
    print(f"\nColor {color}: {len(group)} terms; pairwise commuting = {group_is_pairwise_commuting(group)}")
    for pauli_string, coefficient in group:
        print(f"  {pauli_string:4s}  {format_dense_pauli(pauli_string):15s}  {coefficient:+.10e}")

## 4. Compare With `reorder_paulis`

This is the ordering method already exposed by QHAT as `ordering_method="group_evolve_greedy"`.

In [ ]:
grouped_order = flatten_groups(groups)
qhat_order = list(reorder_paulis(terms, "group_evolve_greedy").items())

print("Order from explicit color groups:")
for index, (pauli_string, _) in enumerate(grouped_order):
    print(f"{index:2d}: {format_dense_pauli(pauli_string)}")

print("\nOrder from reorder_paulis(..., 'group_evolve_greedy'):")
for index, (pauli_string, _) in enumerate(qhat_order):
    print(f"{index:2d}: {format_dense_pauli(pauli_string)}")

## 5. What This Teaches

The naive order count for 14 non-identity terms is `14!`. But H2/STO-3G has only 16 noncommuting pairs out of 91 total pairs. Graph coloring finds two commuting groups. If each group is treated as one contiguous block, there are only `2!` block orders to inspect. If you allow interleavings between the two color groups while ignoring swaps inside each color, the search is still far smaller than `14!`.

This notebook is not yet the full optimizer. It is the conceptual bridge from raw Pauli strings to commuting groups.